In [1]:
!pip install sktime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you h

In [2]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate

In [3]:
import pandas as pd, numpy as np
BASE   = "/kaggle/input/datasets/kimberlybertoli/aircraft-pipeline"
labels = pd.read_csv(f"{BASE}/labels.csv", index_col="Master Index")

In [4]:
import numpy as np
X   = np.load(f"{BASE}/sequences.npy", mmap_mode="r")  # (N, 9212, 15)
idx = np.load(f"{BASE}/seq_indices.npy")
y     = labels.loc[idx, "rul_2d"].values   # swap for rul_5d / rul_10d
split = labels.loc[idx, "split"].values
X_train, y_train = X[split=="train"], y[split=="train"]
X_val,   y_val   = X[split=="val"],   y[split=="val"]
X_test,  y_test  = X[split=="test"],  y[split=="test"]

In [5]:
X_train = X_train.transpose(0, 2, 1)  # (N, 15, 9212)
X_val   = X_val.transpose(0, 2, 1)
X_test  = X_test.transpose(0, 2, 1)

In [6]:
print("Train:", np.bincount(y_train.astype(int)))
print("Val  :", np.bincount(y_val.astype(int)))
print("Test :", np.bincount(y_test.astype(int)))

Train: [5522 8156]
Val  : [1183 1748]
Test : [1184 1747]


In [7]:
rocket = MiniRocketMultivariate()
rocket.fit(X_train[:200])   # safe subset fit

MiniRocketMultivariate()

In [8]:
import os, gc
import numpy as np

OUT_DIR = "/kaggle/working/rocket_features"
os.makedirs(OUT_DIR, exist_ok=True)

def save_chunks(X_mmap, y, rocket, split_name, chunk_size=500):
    n = len(X_mmap)
    chunk_starts = []

    for i in range(0, n, chunk_size):
        end = min(i + chunk_size, n)
        print(f"[{split_name}] chunk {i}–{end}  ({end-i} flights)")

        # Force-load this slice from memmap into RAM, then transform
        X_chunk = np.array(X_mmap[i:end], dtype=np.float32)
        y_chunk = y[i:end]

        feat = rocket.transform(X_chunk)   # (chunk, n_features)

        np.save(f"{OUT_DIR}/{split_name}_X_{i:06d}.npy", feat)
        np.save(f"{OUT_DIR}/{split_name}_y_{i:06d}.npy", y_chunk)

        chunk_starts.append(i)
        del feat, X_chunk, y_chunk
        gc.collect()
        print(f"  Saved to {OUT_DIR}/{split_name}_X_{i:06d}.npy")

    # Save a manifest so you know exactly which chunks exist
    np.save(f"{OUT_DIR}/{split_name}_chunks.npy", np.array(chunk_starts))
    print(f"\n[{split_name}] Done. {len(chunk_starts)} chunks saved.")
    print(f"Manifest: {OUT_DIR}/{split_name}_chunks.npy")


# Run for each split
save_chunks(X_train, y_train, rocket, "train", chunk_size=500)
save_chunks(X_val,   y_val,   rocket, "val",   chunk_size=500)
save_chunks(X_test,  y_test,  rocket, "test",  chunk_size=500)

[train] chunk 0–500  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_000000.npy
[train] chunk 500–1000  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_000500.npy
[train] chunk 1000–1500  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_001000.npy
[train] chunk 1500–2000  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_001500.npy
[train] chunk 2000–2500  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_002000.npy
[train] chunk 2500–3000  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_002500.npy
[train] chunk 3000–3500  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_003000.npy
[train] chunk 3500–4000  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_003500.npy
[train] chunk 4000–4500  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_004000.npy
[train] chunk 4500–5000  (500 flights)
  Saved to /kaggle/working/rocket_features/train_X_004500

In [9]:
# LOAD & ALIGN TABULAR FEATURES
# Must match exactly the order of idx used during chunking
import numpy as np
import pandas as pd

FEAT_DIR = "/kaggle/working/rocket_features"   # where chunks were saved
BASE     = "/kaggle/input/datasets/kimberlybertoli/aircraft-pipeline"

# reload the same idx and split arrays used during chunking
# These were derived from seq_indices.npy + labels.csv in your chunking notebook
idx   = np.load(f"{BASE}/seq_indices.npy")
labels = pd.read_csv(f"{BASE}/labels.csv", index_col="Master Index")
split  = labels.loc[idx, "split"].values        # same order as X_train/val/test

# load tabular features
tabular = pd.read_csv(
    f"{BASE}/tabular_features_scaled.csv",
    index_col="Master Index"
)

feat_cols = [c for c in tabular.columns if "__" in c]
print(f"Tabular feature columns : {len(feat_cols)}")

# alignment check — every idx must exist in tabular
missing = set(idx) - set(tabular.index)
print(f"idx entries missing from tabular: {len(missing)}")
if missing:
    print("  WARNING — these flights have no tabular row:", list(missing)[:5])

# align to idx order (same row order as sequences / chunks)
tabular_aligned = tabular.reindex(idx)          # .loc raises on duplicates; reindex is safe
nan_rows = tabular_aligned[feat_cols].isna().all(axis=1).sum()
print(f"Rows entirely NaN (missing flights): {nan_rows}")

# Fill NaNs — missing rows become 0 vectors, isolated NaNs filled with 0
tabular_aligned[feat_cols] = tabular_aligned[feat_cols].fillna(0)

# split — same mask logic as chunking notebook
T_train = tabular_aligned.loc[split == "train", feat_cols].values.astype(np.float32)
T_val   = tabular_aligned.loc[split == "val",   feat_cols].values.astype(np.float32)
T_test  = tabular_aligned.loc[split == "test",  feat_cols].values.astype(np.float32)

print(f"\nT_train : {T_train.shape}")
print(f"T_val   : {T_val.shape}")
print(f"T_test  : {T_test.shape}")

# sanity check row counts against saved y arrays
y_train = np.load(f"{FEAT_DIR}/train_y_000000.npy")   # first chunk just for length ref
chunks_train = np.load(f"{FEAT_DIR}/train_chunks.npy")
n_train_rocket = sum(
    len(np.load(f"{FEAT_DIR}/train_y_{i:06d}.npy")) for i in chunks_train
)
assert T_train.shape[0] == n_train_rocket, \
    f"Row mismatch: tabular train={T_train.shape[0]}  rocket train={n_train_rocket}"
print(f"\n✓ Row counts match for train ({T_train.shape[0]})")

# ── Step 7: save so you never rerun this
np.save(f"{FEAT_DIR}/train_T.npy", T_train)
np.save(f"{FEAT_DIR}/val_T.npy",   T_val)
np.save(f"{FEAT_DIR}/test_T.npy",  T_test)
print(f"\nSaved train_T / val_T / test_T  to {FEAT_DIR}/")

Tabular feature columns : 297
idx entries missing from tabular: 0
Rows entirely NaN (missing flights): 0

T_train : (13678, 297)
T_val   : (2931, 297)
T_test  : (2931, 297)

✓ Row counts match for train (13678)

Saved train_T / val_T / test_T  to /kaggle/working/rocket_features/
